## Load Input Data

In [ ]:
import pandas as pd

df = pd.read_csv("evaluation_data.csv")

# check for column names: question_text, student_answer, gold_answer, maximal_score, human_score, human_feedback
df.head()

## Call LLMs

In [ ]:
from backend.app.services.exam_simulator import LLMHandler
import json

# TODO: Write prompt for evaluation
prompt = """Bitte vergebe bis zu 5 Punkte für die folgende Studentenantwort auf die gegebene Frage. Für die Orientierung bekommst du eine bereitgestellte Musterlösung.
Vergebe die Punkte folgendermaßen:

Inhaltliche Korrektheit
- Wurden die Antwort komplett richtig und vollständig beantwortet, vergebe 5 Punkte
- Wenn nicht, dann vergebe die erreichten Punkte prozentual

Gebe zusätzlich Verbesserungsvorschläge
- Was sollte der Student inhaltlich besser machen?
- Welche Schlüsselpunkte der Musterlösung wurden getroffen?
- Welche Aussagen sind falsch oder unvollständig?

Gib am Ende ein knappes Gesamtrating (0-5 Punkte). Gerundet auf eine Nachkommastelle

Wichtige Regel: Die gesamte Ausgabe muss in deutscher Sprache erfolgen.

Frage:
{question}

Studentenantwort:
{student_answer}

Musterlösung:
{correct_answer}

Für diese Frage gibt es maximal {max_score} Punkte.

<Antwortformat>
```json
{{
"feedback_content": "<Hier antwortest du auf die Antwort des Studenten und gibts ihm Feedback entsprechend der Analysepunkte. Die Antwort ist ein Text, es gibt keine JSON-Struktur>",
"statement": "<Hier gehst du kurz auf die Antwort des Studenten ein.>",
"overall_rating": "<Gesamtrating (0-5 Punkte)>"
}}```
"""

In [ ]:
def _results_current_model(llm: LLMHandler, prompt: str, df: pd.DataFrame) -> pd.DataFrame:
  llm_results = []
  for _, row in df.iterrows():
    prompt_filled = prompt.format(
      question=row['question_text'],
      student_answer=row['student_answer'],
      correct_answer=row['gold_answer'],
      max_score=row['maximal_score']
    )
    answer = llm.call_llm(prompt=prompt_filled)
    result = json.loads(answer)
    result["question"] = row['question_text']
    result["student_answer"] = row['student_answer']
    result["correct_answer"] = row['gold_answer']
    result["max_score"] = row['maximal_score']
    result["human_score"] = row['human_score']
    result["human_feedback"] = row['human_feedback']
    llm_results.append(result)
    return pd.DataFrame(llm_results)

In [ ]:
models = ["gemma3:27b", "gemma3:4b", "deepseek-r1:8b", "llama3.1:latest", "llava:latest", "command-r7b:latest", "mixtral:latest",
         "mistral-small3.1:latest", "llama3.3:latest", "phi4:latest"]

dfs = []

for model in models:
  llm_handler = LLMHandler(model=model)

  df_model = _results_current_model(llm_handler, prompt, df)
  dfs.append(df_model)
  
final_df = pd.concat(dfs, keys=models, names=['model', 'row'])

## Evaluate LLM Answers

## Comparison and metrics

## Final decision